In [1]:
import os
import warnings

import mne
import numpy as np

warnings.filterwarnings("ignore")

In [4]:
files = [x for x in os.listdir("./data/source") if "TI" in x]
persons = [x[:2] for x in files]

list_epochs = []
for p in persons:
    epoch = mne.read_epochs(f"./data/source/{p}_TI_epochsICA.fif", verbose=False)
    list_epochs.append(epoch)

list_epochs[13] = list_epochs[13].filter(1, 30, verbose=False)

list_labels = []
for id, epochs in enumerate(list_epochs):
    list_labels.append(epochs.events[:, -1] - 2)

In [5]:
# get sampling rate
sampling_rate = list_epochs[0].info["sfreq"]
print(sampling_rate)

1000.0


In [21]:
len(list_labels), len(persons)

(22, 22)

In [23]:
list_labels[0].shape

(60,)

In [31]:
import os

import matplotlib.pyplot as plt

os.makedirs("./data/out/", exist_ok=True)

for i, epochs in enumerate(list_epochs[:1]):
    p = persons[i]

    freqs = np.logspace(*np.log10([1, 30]), num=30)
    n_cycles = freqs / 2.0
    tfr = mne.time_frequency.tfr_morlet(
        epochs, freqs=freqs, n_cycles=n_cycles, return_itc=False, average=False, n_jobs=8
    )
    power = tfr.data
    power = (power - power.min(axis=(0, 1, 3), keepdims=True)) / (
        power.max(axis=(0, 1, 3), keepdims=True) - power.min(axis=(0, 1, 3), keepdims=True)
    )

    for slice_idx in range(power.shape[0]):
        for channel in range(power.shape[1]):
            spectrogram = power[slice_idx, channel]
            plt.imsave(f"./data/out/{p}_spec_{slice_idx}_c{channel}.png", spectrogram)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  10 tasks      | elapsed:   14.9s
[Parallel(n_jobs=8)]: Done  64 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done 127 out of 127 | elapsed:  1.9min finished


Not setting metadata


KeyboardInterrupt: 